# Local Intrinsic Dimensionality as an Unsupervised Quality Signal for Retrieval Indices

## Motivation

In real-world enterprise settings, search/retrieval/RAG indices are built on text distributions that may differ significantly from the training data of the embedder. Ground truth labels are often absent or expensive. This notebook simulates that scenario:

1. **In-distribution (IND)**: **NQ** (Natural Questions) — `all-MiniLM-L6-v2` was fine-tuned on MS MARCO and similar open-QA; NQ is close in domain.
2. **Out-of-distribution (OOD)**: **NFCorpus** — medical/nutritional retrieval, very different from the model's training mix.
3. Mix both, compute retrieval quality (nDCG@10), compute **per-sample local Intrinsic Dimensionality** via `skdim`, filter out low-ID queries, re-evaluate.

**Key insight**: embeddings of OOD text occupy a degenerate (low-dimensional) neighbourhood — the model squashes them together rather than spreading them discriminatively. Local ID captures this geometrically, with **no labels required**.

---
Relation to `intrinsic_dim.py`:
- `compute_intrinsic_dim_local()` returns **summary statistics** per corpus (mean, std, quantiles).
- For **per-sample filtering** we call `estimator.fit_transform_pw()` directly (Section 5).


## 0. Install Dependencies

In [ ]:
# !pip install -q beir sentence-transformers skdim faiss-cpu pytrec_eval-terrier tqdm matplotlib scipy

## 1. Imports & Config

In [ ]:
import os
import random
import logging
import pathlib
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import skdim
import matplotlib.pyplot as plt
from scipy.stats import pointbiserialr, mannwhitneyu
from tqdm.auto import tqdm

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models as beir_models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.WARNING)


@dataclass
class Config:
    # --- Datasets ---
    ind_dataset: str = "nq"          # in-distribution (trained on MS MARCO / open-QA)
    ood_dataset: str = "nfcorpus"    # out-of-distribution (medical domain)
    max_ind_queries: int = 3000      # NQ test has ~3452; cap for speed
    max_ood_queries: int = 323       # NFCorpus test has 323; keep all
    max_ind_corpus: int = 50000      # NQ corpus is ~2.7M; cap
    max_ood_corpus: int = 3633       # NFCorpus corpus is ~3.6k; keep all
    data_dir: str = "datasets"
    # --- Embedder ---
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    batch_size: int = 256
    # --- Local ID ---
    # Note: compute_intrinsic_dim_local() in intrinsic_dim.py returns SUMMARY stats.
    # For per-sample filtering we call fit_transform_pw() directly (see Section 5).
    n_neighbors: int = 30            # k for neighbourhood
    id_estimators: list = field(default_factory=lambda: ["MLE", "MOM"])
    # --- Filtering ---
    filter_percentile: float = 25.0  # remove bottom X% by local_id_mean
    # --- Fine-tuning (optional, Section 12) ---
    do_finetune: bool = False
    finetune_epochs: int = 1
    finetune_batch_size: int = 64


cfg = Config()
pathlib.Path(cfg.data_dir).mkdir(exist_ok=True)
random.seed(42)
np.random.seed(42)
print(cfg)

## 2. Load BEIR Datasets

In [ ]:
def load_beir_dataset(name: str, split: str = "test", data_dir: str = "datasets"):
    """Download (if needed) and load a BEIR dataset."""
    dataset_path = os.path.join(data_dir, name)
    if not os.path.exists(dataset_path):
        url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip"
        util.download_and_unzip(url, data_dir)
    corpus, queries, qrels = GenericDataLoader(data_folder=dataset_path).load(split=split)
    return corpus, queries, qrels


print("Loading IND dataset: NQ ...")
ind_corpus, ind_queries, ind_qrels = load_beir_dataset(cfg.ind_dataset, data_dir=cfg.data_dir)

print("Loading OOD dataset: NFCorpus ...")
ood_corpus, ood_queries, ood_qrels = load_beir_dataset(cfg.ood_dataset, data_dir=cfg.data_dir)

print(f"IND raw : {len(ind_corpus):,} docs, {len(ind_queries):,} queries")
print(f"OOD raw : {len(ood_corpus):,} docs, {len(ood_queries):,} queries")

## 3. Sub-sample to Tractable Size

In [ ]:
def subsample_dataset(corpus, queries, qrels, max_q, max_c, rng_seed=42):
    """Keep at most max_q queries with qrels and max_c corpus docs.
    Relevant documents are always retained.
    """
    rng = random.Random(rng_seed)
    valid_qids = [q for q in queries if q in qrels and len(qrels[q]) > 0]
    sampled_qids = rng.sample(valid_qids, min(max_q, len(valid_qids)))
    queries_sub = {q: queries[q] for q in sampled_qids}
    qrels_sub   = {q: qrels[q]   for q in sampled_qids}

    relevant_cids = set(did for rels in qrels_sub.values() for did in rels)
    extra_cids = [c for c in corpus if c not in relevant_cids]
    rng.shuffle(extra_cids)
    keep_cids = list(relevant_cids) + extra_cids[:max(0, max_c - len(relevant_cids))]
    corpus_sub = {c: corpus[c] for c in keep_cids if c in corpus}
    return corpus_sub, queries_sub, qrels_sub


ind_corpus_s, ind_queries_s, ind_qrels_s = subsample_dataset(
    ind_corpus, ind_queries, ind_qrels, cfg.max_ind_queries, cfg.max_ind_corpus)
ood_corpus_s, ood_queries_s, ood_qrels_s = subsample_dataset(
    ood_corpus, ood_queries, ood_qrels, cfg.max_ood_queries, cfg.max_ood_corpus)

print(f"IND sub-sampled : {len(ind_corpus_s):,} docs, {len(ind_queries_s):,} queries")
print(f"OOD sub-sampled : {len(ood_corpus_s):,} docs, {len(ood_queries_s):,} queries")
print(f"Total queries   : {len(ind_queries_s) + len(ood_queries_s):,}")

## 4. Embed All Queries

We embed **both IND and OOD queries in one pass** so that the local neighbourhood
used for ID estimation reflects the *mixed* geometry — the realistic scenario where
an organisation's index contains documents from diverse, unknown sub-distributions.

In [ ]:
embedder = SentenceTransformer(cfg.model_name)

ind_q_ids   = list(ind_queries_s.keys())
ind_q_texts = [ind_queries_s[q] for q in ind_q_ids]
ood_q_ids   = list(ood_queries_s.keys())
ood_q_texts = [ood_queries_s[q] for q in ood_q_ids]

all_q_ids    = ind_q_ids + ood_q_ids
all_q_texts  = ind_q_texts + ood_q_texts
all_q_labels = ["IND"] * len(ind_q_ids) + ["OOD"] * len(ood_q_ids)

print(f"Embedding {len(all_q_texts):,} queries with {cfg.model_name} ...")
all_q_embs = embedder.encode(
    all_q_texts,
    batch_size=cfg.batch_size,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
ind_q_embs = all_q_embs[:len(ind_q_ids)]
ood_q_embs = all_q_embs[len(ind_q_ids):]
print(f"All query embeddings: {all_q_embs.shape}  (dim={all_q_embs.shape[1]})")

## 5. Per-Sample Local Intrinsic Dimensionality

The helper `compute_intrinsic_dim_local` in `intrinsic_dim.py` computes summary
statistics over a corpus. For **per-sample filtering** we call
`estimator.fit_transform_pw()` directly — this returns one scalar per embedding
representing the local manifold dimensionality around that point.

Lower local ID in a mixed-distribution pool indicates the embedding sits in a
collapsed region — likely OOD / low retrieval quality.

In [ ]:
def compute_local_id_per_sample(
    embeddings: np.ndarray,
    estimator_names: list[str] = ["MLE", "MOM"],
    n_neighbors: int = 30,
    n_jobs: int = -1,
) -> pd.DataFrame:
    """Return a DataFrame with one per-sample local ID column per estimator."""
    emb = embeddings.astype(np.float32)
    results = {}
    for name in tqdm(estimator_names, desc="Local ID estimators"):
        est  = getattr(skdim.id, name)()
        dims = est.fit_transform_pw(emb, n_neighbors=n_neighbors, n_jobs=n_jobs)
        results[f"local_id_{name}"] = dims
    return pd.DataFrame(results)


print(f"Computing local ID on {len(all_q_embs):,} mixed queries (k={cfg.n_neighbors}) ...")
local_id_df = compute_local_id_per_sample(
    all_q_embs,
    estimator_names=cfg.id_estimators,
    n_neighbors=cfg.n_neighbors,
    n_jobs=-1,
)
local_id_df["query_id"]      = all_q_ids
local_id_df["label"]         = all_q_labels
local_id_df["local_id_mean"] = local_id_df[
    [f"local_id_{e}" for e in cfg.id_estimators]
].mean(axis=1)

print("\nLocal ID summary by group (expected: OOD < IND):")
print(
    local_id_df.groupby("label")[["local_id_mean"] + [f"local_id_{e}" for e in cfg.id_estimators]]
    .describe().T
)

## 6. Visualise Local ID Distribution (IND vs OOD)

In [ ]:
fig, axes = plt.subplots(1, len(cfg.id_estimators), figsize=(6 * len(cfg.id_estimators), 4))
if len(cfg.id_estimators) == 1:
    axes = [axes]

for ax, est in zip(axes, cfg.id_estimators):
    col = f"local_id_{est}"
    for label, grp in local_id_df.groupby("label"):
        ax.hist(grp[col].dropna(), bins=40, alpha=0.6, label=label, density=True)
    ax.set_title(f"Local ID ({est})")
    ax.set_xlabel("Local Intrinsic Dimension")
    ax.set_ylabel("Density")
    ax.legend()

plt.suptitle(
    "Local ID Distribution: IND (NQ) vs OOD (NFCorpus) queries\n"
    "Expected: OOD distribution is shifted LEFT (lower dimensionality)",
    y=1.04,
)
plt.tight_layout()
plt.savefig("local_id_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Retrieval Evaluation — Full Mixed Dataset

IND and OOD have separate corpora, so we evaluate them independently and aggregate
with a query-count weighted nDCG@10 to get the overall mixed-index quality.

In [ ]:
def evaluate_retrieval(corpus, queries, qrels, model, batch_size=256, k_values=[1, 3, 5, 10, 100]):
    """Run dense retrieval. `model` can be a name string or a SentenceTransformer instance."""
    beir_model = DRES(beir_models.SentenceBERT(model), batch_size=batch_size)
    retriever  = EvaluateRetrieval(beir_model, score_function="dot", k_values=k_values)
    results    = retriever.retrieve(corpus, queries)
    ndcg, _map, recall, precision = retriever.evaluate(qrels, results, k_values=k_values)
    return ndcg, results


def weighted_ndcg(ndcg_a, n_a, ndcg_b, n_b, k=10):
    key = f"NDCG@{k}"
    return (ndcg_a[key] * n_a + ndcg_b[key] * n_b) / (n_a + n_b)

In [ ]:
print("=== Evaluating IND (NQ) — full ===")
ndcg_ind_full, results_ind_full = evaluate_retrieval(
    ind_corpus_s, ind_queries_s, ind_qrels_s, cfg.model_name, cfg.batch_size)

print("\n=== Evaluating OOD (NFCorpus) — full ===")
ndcg_ood_full, results_ood_full = evaluate_retrieval(
    ood_corpus_s, ood_queries_s, ood_qrels_s, cfg.model_name, cfg.batch_size)

mixed_ndcg10_full = weighted_ndcg(
    ndcg_ind_full, len(ind_queries_s),
    ndcg_ood_full, len(ood_queries_s),
)
print("\n--- Full Mixed Dataset ---")
print(f"IND nDCG@10  : {ndcg_ind_full['NDCG@10']:.4f}  (n={len(ind_queries_s)})")
print(f"OOD nDCG@10  : {ndcg_ood_full['NDCG@10']:.4f}  (n={len(ood_queries_s)})")
print(f"Mixed nDCG@10: {mixed_ndcg10_full:.4f}  (n={len(ind_queries_s)+len(ood_queries_s)})")

## 8. Filter Low Local-ID Queries

In [ ]:
threshold = np.percentile(local_id_df["local_id_mean"].dropna(), cfg.filter_percentile)
print(f"Local ID threshold ({cfg.filter_percentile}th percentile): {threshold:.4f}")

kept_mask   = local_id_df["local_id_mean"] >= threshold
filtered_df = local_id_df[kept_mask].copy()
removed_df  = local_id_df[~kept_mask].copy()

print(f"Kept   : {kept_mask.sum():,} queries ({kept_mask.mean()*100:.1f}%)")
print(f"Removed: {(~kept_mask).sum():,} queries ({(~kept_mask).mean()*100:.1f}%)")
print("\nRemoved breakdown by label:")
print(removed_df["label"].value_counts())
print("\nKept breakdown by label:")
print(filtered_df["label"].value_counts())

## 9. Re-Evaluate on Filtered Query Set

In [ ]:
kept_ind_qids = set(filtered_df[filtered_df["label"] == "IND"]["query_id"])
kept_ood_qids = set(filtered_df[filtered_df["label"] == "OOD"]["query_id"])

ind_queries_filt = {q: ind_queries_s[q] for q in kept_ind_qids if q in ind_queries_s}
ind_qrels_filt   = {q: ind_qrels_s[q]   for q in kept_ind_qids if q in ind_qrels_s}
ood_queries_filt = {q: ood_queries_s[q] for q in kept_ood_qids if q in ood_queries_s}
ood_qrels_filt   = {q: ood_qrels_s[q]   for q in kept_ood_qids if q in ood_qrels_s}

print(f"Filtered IND: {len(ind_queries_filt)} / {len(ind_queries_s)} queries kept")
print(f"Filtered OOD: {len(ood_queries_filt)} / {len(ood_queries_s)} queries kept")

ndcg_ind_filt = ndcg_ood_filt = None

if ind_queries_filt:
    print("\n=== Evaluating IND (filtered) ===")
    ndcg_ind_filt, _ = evaluate_retrieval(
        ind_corpus_s, ind_queries_filt, ind_qrels_filt, cfg.model_name, cfg.batch_size)

if ood_queries_filt:
    print("\n=== Evaluating OOD (filtered) ===")
    ndcg_ood_filt, _ = evaluate_retrieval(
        ood_corpus_s, ood_queries_filt, ood_qrels_filt, cfg.model_name, cfg.batch_size)

In [ ]:
n_filt = len(ind_queries_filt) + len(ood_queries_filt)
if ndcg_ind_filt and ndcg_ood_filt:
    mixed_filt = weighted_ndcg(ndcg_ind_filt, len(ind_queries_filt),
                               ndcg_ood_filt, len(ood_queries_filt))
elif ndcg_ind_filt:
    mixed_filt = ndcg_ind_filt["NDCG@10"]
elif ndcg_ood_filt:
    mixed_filt = ndcg_ood_filt["NDCG@10"]
else:
    mixed_filt = float("nan")

rows = [
    {"Stage": "Full mixed",
     "n_queries": len(ind_queries_s) + len(ood_queries_s),
     "IND nDCG@10": ndcg_ind_full["NDCG@10"],
     "OOD nDCG@10": ndcg_ood_full["NDCG@10"],
     "Mixed nDCG@10": mixed_ndcg10_full},
    {"Stage": f"Filtered (bottom {cfg.filter_percentile}% local ID removed)",
     "n_queries": n_filt,
     "IND nDCG@10": ndcg_ind_filt["NDCG@10"] if ndcg_ind_filt else float("nan"),
     "OOD nDCG@10": ndcg_ood_filt["NDCG@10"] if ndcg_ood_filt else float("nan"),
     "Mixed nDCG@10": mixed_filt},
]
summary_df = pd.DataFrame(rows)
print("\n====== RESULTS SUMMARY ======")
print(summary_df.to_string(index=False))
summary_df.to_csv("results_summary.csv", index=False)

## 10. Visualise: Local ID vs Per-Query Hit@10

In [ ]:
def hit_at_k(results: dict, qrels: dict, k: int = 10) -> dict:
    """Binary hit@k per query."""
    hits = {}
    for qid, doc_scores in results.items():
        if qid not in qrels:
            continue
        relevant  = set(qrels[qid].keys())
        top_k_ids = {d for d, _ in sorted(doc_scores.items(), key=lambda x: -x[1])[:k]}
        hits[qid] = int(bool(relevant & top_k_ids))
    return hits


all_hits = {**hit_at_k(results_ind_full, ind_qrels_s),
            **hit_at_k(results_ood_full, ood_qrels_s)}
local_id_df["hit@10"] = local_id_df["query_id"].map(all_hits)

print("Mean local ID by label x hit@10:")
print(local_id_df.groupby(["label", "hit@10"])["local_id_mean"].mean().unstack())

In [ ]:
rng_j  = np.random.default_rng(0)
colors = {"IND": "#2196F3", "OOD": "#F44336"}

fig, ax = plt.subplots(figsize=(9, 5))
for label, grp in local_id_df.dropna(subset=["hit@10"]).groupby("label"):
    jitter = rng_j.uniform(-0.04, 0.04, len(grp))
    ax.scatter(grp["local_id_mean"], grp["hit@10"] + jitter,
               alpha=0.20, s=10, color=colors[label], label=label)

ax.axvline(threshold, color="black", ls="--", lw=1.5,
           label=f"Threshold ({cfg.filter_percentile}p = {threshold:.2f})")
ax.set_xlabel("Mean Local Intrinsic Dimension")
ax.set_ylabel("Hit@10 (jittered for visibility)")
ax.set_title("Local ID vs Retrieval Hit@10\n(IND=NQ | OOD=NFCorpus)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig("local_id_vs_hit.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Statistical Analysis: Does Low Local ID Predict Retrieval Failure?

In [ ]:
valid = local_id_df.dropna(subset=["local_id_mean", "hit@10"])

corr, pval = pointbiserialr(valid["hit@10"], valid["local_id_mean"])
print(f"Point-biserial r (local_id_mean vs hit@10): r={corr:.4f}, p={pval:.4e}")
print("  Positive r => higher local ID correlates with better retrieval")

hit1 = valid[valid["hit@10"] == 1]["local_id_mean"]
hit0 = valid[valid["hit@10"] == 0]["local_id_mean"]
stat, mwu_p = mannwhitneyu(hit1, hit0, alternative="greater")
print(f"\nMann-Whitney U (hit=1 > hit=0 in local ID): U={stat:.0f}, p={mwu_p:.4e}")

print("\nMean / median local ID by label x hit@10:")
print(valid.groupby(["label", "hit@10"])["local_id_mean"].agg(["mean", "median", "count"]))

In [ ]:
# Sweep filtering aggressiveness without re-running full retrieval.
# Proxy = average hit@10 among kept queries.
percentiles = np.arange(0, 75, 5)
thresholds  = np.percentile(local_id_df["local_id_mean"].dropna(), percentiles)

sweep_rows = []
for p, thr in zip(percentiles, thresholds):
    kept = local_id_df[local_id_df["local_id_mean"] >= thr].dropna(subset=["hit@10"])
    if len(kept) == 0:
        continue
    sweep_rows.append({
        "pct_removed": p,
        "pct_kept": 100 - p,
        "n_kept": len(kept),
        "hit@10_mixed": kept["hit@10"].mean(),
        "hit@10_IND":   kept[kept["label"] == "IND"]["hit@10"].mean(),
        "hit@10_OOD":   kept[kept["label"] == "OOD"]["hit@10"].mean(),
    })

sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df.to_string(index=False))
sweep_df.to_csv("threshold_sweep.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep_df["pct_kept"], sweep_df["hit@10_mixed"], marker="o",  label="Mixed")
ax.plot(sweep_df["pct_kept"], sweep_df["hit@10_IND"],   marker="s",  ls="--", label="IND (NQ)")
ax.plot(sweep_df["pct_kept"], sweep_df["hit@10_OOD"],   marker="^",  ls=":",  label="OOD (NFCorpus)")
ax.invert_xaxis()
ax.set_xlabel("% queries kept  (left = more aggressive filtering)")
ax.set_ylabel("Hit@10 rate  (proxy for nDCG@10)")
ax.set_title("Quality vs Aggressiveness of Local ID Filtering")
ax.legend()
plt.tight_layout()
plt.savefig("threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. (Optional) Fine-tune on Low-ID Samples & Re-Evaluate

Set `cfg.do_finetune = True` to activate.

The removed low-ID queries reveal where the embedder struggles most.
We harvest `(query, relevant_doc)` pairs from their qrels and fine-tune with
`MultipleNegativesRankingLoss` (in-batch negatives).

Expected outcomes:
- **OOD nDCG@10** should improve noticeably.
- **IND nDCG@10** should stay stable or drop slightly (watch for catastrophic forgetting).
- The **local ID distribution** of OOD queries should shift upward after fine-tuning —
  serving as an unsupervised confirmation that the embedder improved.

In [ ]:
if not cfg.do_finetune:
    print("Skipping fine-tuning. Set cfg.do_finetune = True to run this section.")
else:
    low_id_qids = set(removed_df["query_id"])
    train_examples = []

    for qid in low_id_qids:
        for qrels_src, queries_src, corpus_src in [
            (ood_qrels_s, ood_queries_s, ood_corpus_s),
            (ind_qrels_s, ind_queries_s, ind_corpus_s),
        ]:
            if qid not in qrels_src or qid not in queries_src:
                continue
            q_text = queries_src[qid]
            for doc_id, score in qrels_src[qid].items():
                if score > 0 and doc_id in corpus_src:
                    d_text = corpus_src[doc_id].get("text", "")
                    if q_text and d_text:
                        train_examples.append(InputExample(texts=[q_text, d_text]))

    print(f"Fine-tuning on {len(train_examples)} (query, relevant_doc) pairs")

    ft_model     = SentenceTransformer(cfg.model_name)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=cfg.finetune_batch_size)
    train_loss   = losses.MultipleNegativesRankingLoss(ft_model)
    ft_model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=cfg.finetune_epochs,
        warmup_steps=max(10, len(train_loader) // 10),
        show_progress_bar=True,
    )

    print("\n=== Re-evaluating IND (NQ) with fine-tuned model ===")
    ndcg_ind_ft, _ = evaluate_retrieval(
        ind_corpus_s, ind_queries_s, ind_qrels_s, ft_model, cfg.batch_size)

    print("\n=== Re-evaluating OOD (NFCorpus) with fine-tuned model ===")
    ndcg_ood_ft, _ = evaluate_retrieval(
        ood_corpus_s, ood_queries_s, ood_qrels_s, ft_model, cfg.batch_size)

    mixed_ft = weighted_ndcg(
        ndcg_ind_ft, len(ind_queries_s),
        ndcg_ood_ft, len(ood_queries_s),
    )

    fmt = "{:40s} {:>8.4f}  {:>8.4f}  {:>+8.4f}"
    print("\n====== FINE-TUNE RESULTS ======")
    print(f"{'':40s} {'Before':>8s}  {'After':>8s}  {'Delta':>8s}")
    print(fmt.format("IND nDCG@10",
                     ndcg_ind_full['NDCG@10'], ndcg_ind_ft['NDCG@10'],
                     ndcg_ind_ft['NDCG@10'] - ndcg_ind_full['NDCG@10']))
    print(fmt.format("OOD nDCG@10",
                     ndcg_ood_full['NDCG@10'], ndcg_ood_ft['NDCG@10'],
                     ndcg_ood_ft['NDCG@10'] - ndcg_ood_full['NDCG@10']))
    print(fmt.format("Mixed nDCG@10 (weighted)",
                     mixed_ndcg10_full, mixed_ft, mixed_ft - mixed_ndcg10_full))

    # Check if local ID shifts upward after fine-tuning
    print("\nRe-computing local ID with fine-tuned embedder ...")
    all_q_embs_ft = ft_model.encode(
        all_q_texts, batch_size=cfg.batch_size,
        show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
    )
    local_id_ft = compute_local_id_per_sample(
        all_q_embs_ft, estimator_names=cfg.id_estimators,
        n_neighbors=cfg.n_neighbors, n_jobs=-1,
    )
    local_id_ft["label"]         = all_q_labels
    local_id_ft["local_id_mean"] = local_id_ft[
        [f"local_id_{e}" for e in cfg.id_estimators]
    ].mean(axis=1)

    print("\nLocal ID shift after fine-tuning:")
    before = local_id_df.groupby("label")["local_id_mean"].mean().rename("before")
    after  = local_id_ft.groupby("label")["local_id_mean"].mean().rename("after")
    print(pd.concat([before, after], axis=1).assign(delta=lambda d: d["after"] - d["before"]))